# SBERT-powered Lexicon Expansion

In this next stage of the pipeline I use SBERT as a semantic filter to selectivley expand the theory-derived seed-list lexicon across the 3 dimensions (Threat, Urgency & Justification of Extraordinary Measures).

The continuous similarity scores produced by the embeddings require me to set dynamic thresholds for each dimension. In doing this I achieve a context-sensitive & verifiable human-in-the-loop expansion across all 3 dimensions.

## Use of Generative AI
I used generative AI (specifically Claude Sonnet 4.6) in this file help specifically with formatting. Throughout this file I have three different dimensions each split into three different tiers across sections of this notebook and I found that my original setup of format became very difficult to manage very quickly. I explained this problem to the AI and asked it for help to reformat my old approach to categorising and presenting the data into a way that was clearer to read and keep track of, so that I could avoid any mistakes I made from a messy layout, and that new approach is what I've used below.

However, all the logic, including the preprocessing, categorisation, expanding into a lexicon from the literature and examining non-stemmed variants were entirely my own work; AI just helped me with formatting this so I could present it neater and not make avoidable mistakes.

# 0. Global Config

In [ ]:
!pip install nltk fastparquet xlsxwriter

from typing import Dict, Set, List
import os
import pandas as pd
import re
from collections import defaultdict, Counter
import numpy as np
from numpy.linalg import norm
import math
import nltk
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
from nltk.corpus import stopwords, words
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import fastparquet
import matplotlib.pyplot as plt
from IPython.display import display
import xlsxwriter

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')

In [ ]:
# I chose to setup all the configs for this file at the top things can be easily tweaked & adjusted at will
lexicon_expansion_config= {
    # file paths
    "seed_lexicon_path": "securitisation_lexicon.xlsx",
    "corpus_file_prefix": "manifesto_corpus",
    "sbert_scores_file_prefix": "sbert_lexicon_scores",

    # short tokens that i define as valid in bigram contexts even though len <= 2
    "short_token_whitelist": {"of", "on", "to"},

    # random seed for reproducibility
    "random_seed": 42,

    # final selected percentile per dimension (used in the main selection pass)
    "final_dimension_percentiles": {
        "threat": 0.99,
        "urgency": 0.995,
        "justification": 0.995,
    },

    # percentile ranges I use during the per-dimension exploration phase
    "threat_exploration_percentiles": [0.985, 0.99, 0.992, 0.995],
    "urgency_exploration_percentiles": [0.985, 0.99, 0.992, 0.995],
    "justification_exploration_percentiles": [0.995, 0.997, 0.998, 0.999],
}

# 1. Load Corpus and Seed Lexicons
Now the lexicon set I created in `Securitisation_Lexicons_Construction.ipynb` is loaded in alongside the manifesto corpus to prepare for this expansion.

In [ ]:
# this loads all 3 of my seed lexicons from the excel file into a dict of sets, 1 for each lexicon
def loadin_seed_lexicons():
    seed_lexicon_path= lexicon_expansion_config["seed_lexicon_path"]
    if not os.path.exists(seed_lexicon_path):
        print(f"couldn't find the lexicon @ {seed_lexicon_path}.")
        return {}

    try:
        print(f"loading lexicons from {seed_lexicon_path}...")
        dimension_sheets= pd.read_excel(seed_lexicon_path, sheet_name=['Threat Lexicon', 'Urgency Lexicon', 'Justification Lexicon'])
        seed_lexicons= {
            'threat': set(dimension_sheets['Threat Lexicon'].iloc[:, 0].dropna().str.lower()),
            'urgency': set(dimension_sheets['Urgency Lexicon'].iloc[:, 0].dropna().str.lower()),
            'justification': set(dimension_sheets['Justification Lexicon'].iloc[:, 0].dropna().str.lower())
        }
        print(f"Lexicons loaded!! For each dimension I have: Threat ({len(seed_lexicons['threat'])}), Urgency ({len(seed_lexicons['urgency'])}), Justification ({len(seed_lexicons['justification'])})")
        return seed_lexicons
    except Exception as lex_load_err:
        raise Exception(f"isse loading in lexicon: {lex_load_err}")

seed_lexicons= loadin_seed_lexicons()
print(f"Threat Lexicon: {seed_lexicons['threat']}")
print(f"Urgency Lexicon: {seed_lexicons['urgency']}")
print(f"Justification Lexicon: {seed_lexicons['justification']}")

In [ ]:
# this finds & loads the 1st parquet file matching the corpus prefix in the current directory
def loadin_manifesto_corpus():
    all_files_in_dir= [f_here for f_here in os.listdir('.') if os.path.isfile(f_here)]
    corpus_file_prefix= lexicon_expansion_config["corpus_file_prefix"]
    matching_corpus_files= [f_match for f_match in all_files_in_dir if f_match.startswith(corpus_file_prefix) and f_match.endswith(".parquet")]
    if not matching_corpus_files:
        raise FileNotFoundError(f"no parquet file starting with '{corpus_file_prefix}' found in the current directory.")

    # I taking the first match - if multiple are present, I x2-check the version number
    corpus_file= matching_corpus_files[0]
    print(f"loading corpus from {corpus_file}...")

    try:
        loaded_df= pd.read_parquet(corpus_file)
        print(f"get, got corpus. It has {loaded_df.shape[0]} rows, {loaded_df.shape[1]} columns.")
        return loaded_df
    except Exception as load_corpus_e:
        raise Exception(f"hmm issue loading corpus {corpus_file}: {load_corpus_e}")

manifesto_corpus_df= loadin_manifesto_corpus()
display(manifesto_corpus_df.head())

# 2. Minimal Corpus Cleanup
Most preprocessing has already occured prior to loading in the corpus. However, based on what I've read about SBERT being used in this context, the pipeline will perform better by replacing puncutation (except apostrophies inside words like "don't") with spaces. *Note: Although SBERT is robust to puncutation, token extraction for n-grams isn't.*

No lemmatisiation, stemming, stopword removal or POS-filters are applied though.

In [ ]:
# replaces punctuation with spaces, leaving apostrophes inside words intact (e.g. "don't")
# SBERT is robust to punctuation, but the n-gram extraction later isn't
def normalise_corpus_punctuation(df):
    # this pattern leaves apostrophes between letters alone and replaces everything else
    punctuation_pattern= r"(?<![a-zA-Z])'|'(?![a-zA-Z])|[^\w\s]"

    def clean_sentence_text(text):
        if not isinstance(text, str):
            return text
        cleaned= re.sub(punctuation_pattern, ' ', text)
        return re.sub(r'\s+', ' ', cleaned).strip()

    df['sentence_text']= df['sentence_text'].apply(clean_sentence_text)
    return df

manifesto_corpus_df= normalise_corpus_punctuation(manifesto_corpus_df)

# 3. Unigram Extraction
Firstly I filter the corpus to exclude all words of length <= 2 from the candidate universe. Then, unigrams are extracted.

Stopwords are retained as they still play a role in securitising sentences. SBERT naturally downweight their saliency due to their broad contextual use.

In [ ]:
# builds a mapping of each unigram to the sentence indices it appears in
# (only terms longer than 2 chars are kept)
unigrams_to_sentence_indices= defaultdict(list)

for sentence_idx, row in manifesto_corpus_df.iterrows():
    # each unique term is only processed once per sentence
    unique_tokens_in_sentence= set(row["sentence_text"].lower().split())

    for sent_toks in unique_tokens_in_sentence:
        if len(sent_toks) > 2:
            unigrams_to_sentence_indices[sent_toks].append(sentence_idx)

# i do a quick freq preview for visual inspection
unigram_frequency_list= [(word, len(indices)) for word, indices in unigrams_to_sentence_indices.items()]
unigram_frequency_list.sort(key=lambda x: x[1], reverse=True)

top_unigrams_preview_df= pd.DataFrame(unigram_frequency_list[:5], columns=['Candidate Term', 'Corpus Frequency'])
print("top 5 terms in corpus (could be viable stopword removal candidates)")
display(top_unigrams_preview_df)

# 4. Bi-gram Extraction
Bi-grams are extracted from the corpus firstly.

Then, log-likelihood scores are computed and used as a ranking signal. It preserves rare but meaningful phrases (as securitised language often is), ensuring common bigrams don't dominate.

What's more, it penalises accidental co-occurence, and is stable under the uneven frequency distribution of a manifesto.

*Note that PMI was not used as it would favour bigrams appearing once or twice too strongly, which is methodologically dangerous in manifestos, which can differ in focus points between parties quite significantly at times.*

In [ ]:
english_stopwords= set(stopwords.words("english"))
short_token_whitelist= lexicon_expansion_config["short_token_whitelist"]
bigram_assoc_measures= BigramAssocMeasures()

# i flatten the corpus into a single token list for the bigram finder
all_corpus_tokens= []

for sentence in manifesto_corpus_df["sentence_text"]:
    sent_toks= sentence.lower().split()
    filtered_tokens= [tok for tok in sent_toks if len(tok) > 2 or tok in short_token_whitelist]
    all_corpus_tokens.extend(filtered_tokens)

bigram_finder= BigramCollocationFinder.from_words(all_corpus_tokens)

In [ ]:
# raw frequencies before any filtering, which is useful to see what I'm working with
raw_bigram_freqs= [{"bigram": f"{firstword} {secondword}", "frequency": freq} for (firstword,secondword), freq in bigram_finder.ngram_fd.items()]

bigram_frequency_df= pd.DataFrame(raw_bigram_freqs)
bigram_frequency_df.sort_values("frequency", ascending=False, inplace=True)

print("top 15 bigrams by freq. (unfiltered) are these:")
display(bigram_frequency_df.head(15))

Looking at the above, a few clasic PDF line break reconstruction artefacts stand out (e.g. "the the", "the and"). These are to be expected, and indicate localised sentence boundary noise typical of PDF extraction.

Importantly, this doesn't indicate widespread OCR corruption and won't undermine SBERT useage, as it's robust to function words, repeitition and midly malfomed sentences.

However, bigrams are contextually meaningless and should thus I will remove them from the candidate pool. To mitigate this without overfitting the corpus, I drop bigrams composed entirely of stopwords, which carry only syntactic information. This is safe, as securitising meaning will be conveyed through context words.

In [ ]:
# drop bigrams made entirely of stopwords as they carry no securitising meaning
def is_fully_stopword_bigram(firstword, secondword):
    return firstword in english_stopwords and secondword in english_stopwords

bigram_finder.apply_ngram_filter(is_fully_stopword_bigram)

# now i compute log-likelihood scores for ranking (chosen over PMI to avoid over-inflating rare bigrams)
log_likelihood_scored_bigrams= bigram_finder.score_ngrams(bigram_assoc_measures.likelihood_ratio)

In [ ]:
bigram_log_likelihood_df= pd.DataFrame([{"bigram": f"{firstword} {secondword}", "log_likelihood": score} for (firstword, secondword), score in log_likelihood_scored_bigrams])
bigram_log_likelihood_df.sort_values("log_likelihood", ascending=True, inplace=True)
print("bottom 25 bigrams by log-likelihood (filtered) are  these:")
display(bigram_log_likelihood_df.head(25))

The above bottom 25 log-likelihood candidates are generally viable bigrams. Therefore my decision is to not remove them to avoid corpus overfitting.

At this point, I now have a statistically-ranked candidate space that's filtered of structural junk. It's not yet dimension-specific.

The next step is preparing this list for contextual semantic validation.

To do this, a contextual table containing each unigram and bigram alongside the context they appear in is created. This allows for audiability for later lexicon additions.

This table enables me to perform human-in-the-loop expansion, as terms that are added can be inspected and validated in sentence context.

In [ ]:
# collect only the bigrams that made it through the log-likelihood filter
meaningful_bigram_set= set(bigram_log_likelihood_df['bigram'].values)

# map each meaningful bigram to the sentence indices it appears in
bigrams_to_sentence_indices= defaultdict(list)

for sentence_idx, row in manifesto_corpus_df.iterrows():
    sentence_tokens= row["sentence_text"].lower().split()
    filtered_tokens= [
        tok for tok in sentence_tokens
        if len(tok) > 2 or tok in short_token_whitelist
    ]

    for i in range(len(filtered_tokens) - 1):
        candidate_bigram= f"{filtered_tokens[i]} {filtered_tokens[i + 1]}"
        if candidate_bigram in meaningful_bigram_set:
            bigrams_to_sentence_indices[candidate_bigram].append(sentence_idx)

# now I build an audit table which maps every unigram and bigram to its sentence contexts
audit_rows= []

for term, sentence_ids in unigrams_to_sentence_indices.items():
    for sid in sentence_ids:
        audit_rows.append({"ngram_type": "unigram", "ngram": term, "sentence_id": sid})
for term, sentence_ids in bigrams_to_sentence_indices.items():
    for sid in sentence_ids:
        audit_rows.append({"ngram_type": "bigram", "ngram": term, "sentence_id": sid})

term_context_audit_df= pd.DataFrame(audit_rows)
term_context_audit_df= term_context_audit_df.merge(
    manifesto_corpus_df[['manifesto_id', 'partyname', 'election_year', 'sentence_text']],
    left_on='sentence_id',
    right_index=True
)

# & I export to excel with separate tabs for unigrams and bigrams (just because its easier for me to scan)
audit_output_path= "lexicon_expansion_audit.xlsx"

try:
    with pd.ExcelWriter(audit_output_path, engine='openpyxl') as excel_writer:
        term_context_audit_df[term_context_audit_df['ngram_type'] == 'unigram'].to_excel(excel_writer, sheet_name='Unigram Contexts', index=False)
        term_context_audit_df[term_context_audit_df['ngram_type'] == 'bigram'].to_excel(excel_writer, sheet_name='Bigram Contexts', index=False)

    print(f"great, exported that audit file @ {audit_output_path}")
except Exception as export_e:
    print(f"uh oh, couldn't export. Error is this for tracing: {export_e}")

display(term_context_audit_df.head())

# 5. Generating contextual SBERT embeddings
Each unigram and bigram will now be embedded in sentence contexts and averged into a single term vector.

In [ ]:
# optional: skip the SBERT embedding cells below and load scores from a previously saved file
def load_saved_similarity_scores(location_of_sbert_scores_for_resuming="sbert_lexicon_scores.parquet"):
    if not os.path.exists(lexicon_expansion_config["sbert_scores_file_prefix"]):
        raise FileNotFoundError(f"hmm, similarity score file not found from {location_of_sbert_scores_for_resuming}, check it's actually in this directory.")

    try:
        loaded_scores_df= pd.read_parquet(lexicon_expansion_config["sbert_scores_file_prefix"], engine='fastparquet')
        print(f"alright, great, loaded in {len(loaded_scores_df)} candidate-dimension pairs.")
        return loaded_scores_df
    except Exception as sim_score_load_e:
        raise Exception(f"hm nope, failed to read similarity scores: {sim_score_load_e}")

candidate_similarity_df= load_saved_similarity_scores()

In [ ]:
# # load the SBERT model
# sbert_model= SentenceTransformer("all-mpnet-base-v2")

# # I build one embedding per term by averaging across all its sentence contexts
# term_contextual_embeddings= {}

# grouped_by_term= term_context_audit_df.groupby("ngram")
# for term, group in tqdm(grouped_by_term, desc="building term embeddings"):
#     sentence_contexts= group["sentence_text"].tolist()
#     context_embeddings_for_lexicons= sbert_model.encode(sentence_contexts, show_progress_bar=True)
#     lexicon_term_contextual_embeddings[term]= np.mean(context_embeddings_for_lexicons, axis=0)

# print(f"great, built embeddings for {len(lexicon_term_contextual_embeddings)} candidate terms")

Now that the uni & bi grams have been embedded in sentence contexts, the seed lexicons each be embedded and a centroid vector per dimension will be created. I'll then compare each candidate term to that centroid using computed cosine similarity scores.

In [ ]:
# # embed each seed lexicon and compute a centroid vector per dimension
# dimension_centroid_embeddings= {}

# for dimension, seed_terms in tqdm(seed_lexicons.items(), desc="building centroid embeddings"):
#     seed_term_list= list(seed_terms)
#     seed_term_embeddings= sbert_model.encode(seed_term_list, show_progress_bar=True)
#     dimension_centroid_embeddings[dimension]= np.mean(seed_term_embeddings, axis=0)

# print("good, built centroid embeddings for all dimensions")

In [ ]:
# # compute cosine similarity of each term embedding against each dimension centroid
# def cosine_similarity_score(vec_a, vec_b): # this is just a standard use function, I just re-implemented from https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html logic
#     return np.dot(vec_a, vec_b) / (norm(vec_a) * norm(vec_b))

# similarity_rows= [] # to store what will go in the combined df I create
# for term, term_embedding in tqdm(term_contextual_embeddings.items(), desc="computing cosine similarities"):
#     for dimension, centroid_embedding in dimension_centroid_embeddings.items():
#         sim= cosine_similarity_score(term_embedding, centroid_embedding)
#         similarity_rows.append({"term": term, "dimension": dimension, "cosine_similarity": sim})

# candidate_similarity_df= pd.DataFrame(similarity_rows)

In [ ]:
# # export similarity scores as both csv (for human review) and parquet (for downstream use)
# try:
#     sbert_scores_prefix= lexicon_expansion_config["sbert_scores_file_prefix"]

#     candidate_similarity_df= candidate_similarity_df.sort_values(
#         by=['dimension', 'cosine_similarity'],
#         ascending=[True, False]
#     )

#     candidate_similarity_df.to_csv(f"{sbert_scores_prefix}.csv", index=False)
#     candidate_similarity_df.to_parquet(f"{sbert_scores_prefix}.parquet", index=False, engine='fastparquet')

#     print(f"yep, export done.")
#     print(f"you can find the csv audit file @ {sbert_scores_prefix}.csv")
#     print(f"& parquet @ {sbert_scores_prefix}.parquet")
# except NameError:
#     print("nope, got an error on this one, where candidate_similarity_df is not defined. re-run the similarity computation cell.")
# except Exception as sim_e:
#     print(f"this failed, checkout the error: {sim_e}")

At this point, a semantic space is created. Using this I can now compare term embeddings to their similarity to each dimension. I use the cosine similarity score here as it gives a measure of how strongly an embeddings is aligned with a dimension.

In [ ]:
# I apply a light structural filter to remove OCR artefacts. Note this isn't a linguistic quality filter though
print(f"lines before cleaning are sitting at {len(candidate_similarity_df)}")

# rules: must match a single word or two-word phrase (letters only), no repeated chars (OCR corruption), no digits
def is_structurally_valid_term(term):
    if term is None:
        return False
    term= term.lower().strip()
    if not re.fullmatch(r'[a-z]+(?: [a-z]+)?', term): # the conditions by which I define a term to be valid can be captured through regex best
        return False

    # this bit catches OCR corruption I might have missed in the extraction like "cchhaannggee" or "ssecurity"
    if re.search(r'(.)\1{2,}', term):
        return False
    if re.search(r'\d', term):
        return False
    return True


valid_term_mask= candidate_similarity_df['term'].apply(is_structurally_valid_term)
filtered_similarity_df= candidate_similarity_df[valid_term_mask].copy()

# I save removed terms for a later human audit
lexicon_artefacts_removed_df= candidate_similarity_df[~valid_term_mask].copy()
lexicon_artefacts_removed_df[['term']].drop_duplicates().to_csv("removed_lexicon_terms_for_audit.csv", index=False)
candidate_similarity_df= filtered_similarity_df
print(f"lines after cleaning are {len(candidate_similarity_df)}")
display(candidate_similarity_df.head())

*Note: the term "tleave" (amongst others) is clearly a corrupted artefact, but not an indication of my code failling. I found out this was because SBERT embeds what typically surrounds the term "teave", and as it appears in a highly charged context in the corpus it shows that context dominates surface form.*

*As rare terms were intentionally not supressed (as securitising language can be rare but salient, e.g. in a singular far-right manifesto), this is to be expected.*

*I did try to remove these terms but I found that applying dictionary-based filtering to this corpora is unsuitable; standard dictionaries don't capture modern-day political language, and thus genuinly securitising political language will be dropped at this stage if that technique was applied.*

# 6. Thresholding
The aim now is to determine which of the masssive ~812k candidate terms are sufficiently semantically aligned with a securitisation dimension to warrant human inspection. Obviously this is way too much for me to review, so I use thresholding, and quite a high threshold at that. Doing this allows me to:


*   reduce candidate set to a manageable size
*   preserve recall over precision (ensuring a sufficent quantity of words are available to choose from)
*   highlight meaningful variations between dimensions
*   and importantly, avoid arbitrary top-N truncating


The candidate thresholds for each dimension determine the set of words that are to be manually validated. They're grounded in the internal coherence of the theoretically-derived seed lexicons.

The idea is as follows:
*Given a fixed semantic space anchored by validated seed lexicons, how far outward should I expand before semantic drift becomes unnaceptable?*


In [ ]:
# 1st I pull percentile config and collapse the similarity table to one row per term
dimension_final_percentiles= lexicon_expansion_config["final_dimension_percentiles"]
np.random.seed(lexicon_expansion_config["random_seed"])

# then I assign each term to its most semantically aligned dimension
best_dimension_per_term_df= (candidate_similarity_df.sort_values("cosine_similarity", ascending=False).groupby("term", as_index=False).first())

print("alright, similarity table has been collapsed.")
print(f"there were {len(candidate_similarity_df):,} original rows but")
print(f"now there's {len(best_dimension_per_term_df):,} unique terms")

In [ ]:
# now I annotate seed membership for diagnostics (i don't use this as a filter though)
def term_is_in_seed_lexicon(term, dimension):
    return term in seed_lexicons.get(dimension, set())

best_dimension_per_term_df["is_seed"]= best_dimension_per_term_df.apply(lambda row: term_is_in_seed_lexicon(row["term"], row["dimension"]), axis=1)
print("ok, seed membership is now annotated (for my diagnostic use only)")

For each dimension I compute the intra-seed similarity distribution and take the lower quartile as the acceptance threshold. This gives me different thresholds per dimension, relfecting the varying linguistic coherence between Threat, Urgency and Extraordinary Power as features of securitisation.
let me feasibly review the top candidates.

In [ ]:
# apply dimension-specific percentile thresholds to produce the final candidate sets
final_candidates_per_dim= {}
similarity_thresholds_per_dim= {}

for dim, percentile in dimension_final_percentiles.items():
    current_dim_df= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == dim].copy()

    current_seed_terms= current_dim_df[current_dim_df["is_seed"]]
    current_candidate_terms= current_dim_df[~current_dim_df["is_seed"]]

    # I compute threshold on the candidates only, but seeds are always included regardless
    dim_similarity_threshold= current_candidate_terms["cosine_similarity"].quantile(percentile)
    similarity_thresholds_per_dim[dim]= dim_similarity_threshold

    selected_above_threshold= current_candidate_terms[
        current_candidate_terms["cosine_similarity"] >= dim_similarity_threshold
    ].sort_values("cosine_similarity", ascending=False)

    # my final lexicon for each dim == seeds + candidates above the threshold
    final_dim_candidate_df= pd.concat([current_seed_terms, selected_above_threshold],ignore_index=True).sort_values("cosine_similarity", ascending=False)
    final_candidates_per_dim[dim]= final_dim_candidate_df

    # & these cutoffs decide how much of the ranked list for each dimension is available for inspection
    print(f"\nUpper bound for this dimension is {dim.upper()}")
    print(f"The percentile I chose to apply was {percentile:.3f}")
    print(f"w/ similarity cuttoff {dim_similarity_threshold:.4f}")
    print(f"so overall I added {len(selected_above_threshold):,} candidates for inspection")
    print(f"& the seed terms, which are always kept, are of legnth {len(current_seed_terms):,}")
    print(f"so that brings the total lexicon size to {len(final_dim_candidate_df):,}")

In [ ]:
# this is a visual diagnostic for me to answer the q "where do seed terms sit relative to the applied cutoff?"
# it compares the semantic distribution of seeds against all candidates per dimension
fig, axes= plt.subplots(2, len(dimension_final_percentiles), figsize=(18, 6), sharex="col")
if len(dimension_final_percentiles) == 1:
    axes= axes.reshape(2, 1)

for col, dim in enumerate(dimension_final_percentiles.keys()):
    all_terms_for_dim= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == dim]
    seed_terms_for_dim= all_terms_for_dim[all_terms_for_dim["is_seed"]]

    # top row is just seed terms only
    axes[0, col].hist(seed_terms_for_dim["cosine_similarity"], bins=20, alpha=0.9, label="seed terms")
    axes[0, col].axvline(similarity_thresholds_per_dim[dim], color="red", linestyle="--", linewidth=2, label="applied similarity cutoff")
    axes[0, col].set_title(f"{dim.capitalize()} just for seed terms")
    axes[0, col].set_ylabel("freq.")
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(alpha=0.3)

    # & bottom row all candidates
    axes[1, col].hist(all_terms_for_dim["cosine_similarity"], bins=50, alpha=0.6, label="all candidates")
    axes[1, col].axvline(similarity_thresholds_per_dim[dim], color="red", linestyle="--", linewidth=2, label="applied similarity cutoff")
    axes[1, col].set_title(f"{dim.capitalize()} for all the candidates")
    axes[1, col].set_xlabel("cosine similarity")
    axes[1, col].set_ylabel("freq.")
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### *Interpreting the above*
The above distributions show how the threshold is aligned with the seed terms. The percentiles control how much of the candidate space is to be ispected, and seed terms act as a semantic calibration signal for this.

In [ ]:
# this applies dimension-specific percentile cutoffs to the collapsed similarity table.
# I don't remove seeds but only include them for diagnosing purposes
def apply_dimension_percentile_cutoffs(best_dimension_per_term_df, seed_lexicons, dimension_final_percentiles):
    final_candidates_per_dim= {}
    similarity_thresholds_per_dim= {}
    summary_rows= []

    for dim, percentile in dimension_final_percentiles.items():
        current_dim_df= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == dim].copy()

        dim_similarity_threshold= current_dim_df["cosine_similarity"].quantile(percentile)
        similarity_thresholds_per_dim[dim]= dim_similarity_threshold
        terms_above_threshold= current_dim_df[current_dim_df["cosine_similarity"] >= dim_similarity_threshold].sort_values("cosine_similarity", ascending=False)

        total_seeds= len(seed_lexicons.get(dim, []))
        seeds_in_selection= terms_above_threshold["is_seed"].sum()

        final_candidates_per_dim[dim]= terms_above_threshold

        summary_rows.append({"dimension": dim, "percentile": percentile, "cutoff": round(dim_similarity_threshold, 4),
            "candidates_kept": len(terms_above_threshold), "seeds_retained": int(seeds_in_selection), "seeds_total": int(total_seeds)})

    summary_df= pd.DataFrame(summary_rows)
    return final_candidates_per_dim, similarity_thresholds_per_dim, summary_df


# & here I inspect terms in a narrow band around the cutoff, which is useful for spotting semantic drift
def inspect_cutoff_margin_terms(best_dimension_per_term_df, similarity_thresholds_per_dim, dim, window=20):
    current_dim_df= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == dim].copy()
    dim_similarity_threshold= similarity_thresholds_per_dim[dim]

    # important context: i define delta as the cosine similarity cutoff, so here it's a measure of how close to the cutoff a
    # uni/bigram is. So large +ve delta == semantic robustness + lower risk of inclusion, ~0 == borderline & conceptually ambiguous
    # & -ve delta is just excluded, useful for validation of my method only.
    current_dim_df["delta"]= current_dim_df["cosine_similarity"] - dim_similarity_threshold
    current_dim_df= current_dim_df.sort_values("delta", ascending=False)

    terms_above_cutoff= current_dim_df[(current_dim_df["delta"] >= 0)].tail(window)
    terms_below_cutoff= current_dim_df[(current_dim_df["delta"] < 0)].head(window)

    margin_df= pd.concat([terms_above_cutoff, terms_below_cutoff])
    margin_df= margin_df[["term", "cosine_similarity", "delta", "is_seed"]]
    margin_df["cosine_similarity"]= margin_df["cosine_similarity"].round(4)
    margin_df["delta"]= margin_df["delta"].round(4)

    return margin_df


print("threat cutoff terms")
display(inspect_cutoff_margin_terms(best_dimension_per_term_df, similarity_thresholds_per_dim, dim="threat", window=15))
print("\n urgency cutoff terms")
display(inspect_cutoff_margin_terms(best_dimension_per_term_df, similarity_thresholds_per_dim, dim="urgency", window=15))
print("\n justification of extraordinary power cutoff terms")
display(inspect_cutoff_margin_terms(best_dimension_per_term_df, similarity_thresholds_per_dim, dim="justification", window=15))

### Per-dimension semantic exploration
For me to accept terms into the lexicons they need to be semantically aligned with their corresponding secuiritisation dimension. To do this I experiment with a range of candidate percentiles for each dimension, acknowledging the fact that each dimension is fundamentally different and thus the point at which semantic alignment stops being vialbe will vary.

Therefore, CSVs are generated for each dimension for visual inspection, with each tab in the CSV containing the words encompassed within a given percentile.

Formally, I define semantic drift as:
*the point at which its securitising function can't be justified without additional contextual assumptions not encoded in the term itself*.

I choose the cutoff selection as:
*The point at which the highest percentile p has fewer than 30% of its maginal terms within the inspection windows exhibiting semantic drift.*
After this, I manually vet every term I accept above the cutoff.

In [ ]:
# this reads a manually annotated lexicon expansion excel and returns all terms I marked 'y' for a given dimension
def loadin_vetted_dimension_terms(dimension: str, excel_file: str, sheet_name: str) -> set:
    vetted_df= pd.read_excel(excel_file, sheet_name=sheet_name)
    securitising_rows= vetted_df[vetted_df["securitising?"].str.lower() == "y"]
    return set(securitising_rows["term"].dropna().tolist())

### a. Threat
The below lists produced by the percentiles is a collection of threatening language somewhat semantically similar to the Threat seed lexicon. However, it's important to ensure that accepted terms are threat-related in the securitisation sense, i.e. that it's language that frames a referent object as facing existential descrution or irreversible harm at a collective or system level.

Therefore, to be accepted into the Threat lexicon, the uni/bigram must be:


*   Existential, not routine.
*   Collective, not individual.
*   & plausible in a speech act that elavates the issue to national or civilisational stakes.

Examples from p=0.985 that fail and why:


1.   "with shoplifting" because it describes crime that doesn't imply eisential descruction of a collective referent.
2.   "themselves harm" as its about individual harm, lacking systemic/civilisational scale.


And examples that do:


1.   "violent overthrow" as it's an existential polictical threat
2.   "sexportation" as it's a national-level threat against a group of venrable individuals.

Some bigrams may contain a word in (or related to) the securitisation seed list but are excluded due to being sentence artefacts rather than securitised bigrams. Examples of rejected bigrams containing a securitised "Threat" word in a UK context include:


1.   "Gaza and"
2.   "Iraq to"
3.   "aegis"
4.   "aweapon" (OCR artefact)


And examples of ones accepted are:


1.   "genocide Gaza"
2.   "brutal terrorism"
3.   "aegis of"


In [ ]:
# exploring threat percentiles. Percentile choice controls breadth of candidate pool
# Not ranking evaluation focuses on marginal terms near the cutoff where semantic drift is most likely

threat_exploration_percentiles= lexicon_expansion_config["threat_exploration_percentiles"]
threat_trials_output_path= "threat_percentile_trials.xlsx"

with pd.ExcelWriter(threat_trials_output_path, engine="xlsxwriter") as excel_writer:

    threat_dim_df= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == "threat"].copy()
    threat_seed_terms= threat_dim_df[threat_dim_df["is_seed"]]
    threat_candidate_terms= threat_dim_df[~threat_dim_df["is_seed"]]

    for thr_p in threat_exploration_percentiles:
        percentile_cutoff= threat_candidate_terms["cosine_similarity"].quantile(thr_p)
        candidates_above_cutoff= threat_candidate_terms[threat_candidate_terms["cosine_similarity"] >= percentile_cutoff].sort_values("cosine_similarity", ascending=False)
        threat_trial_df= pd.concat([threat_seed_terms, candidates_above_cutoff], ignore_index=True).sort_values("cosine_similarity", ascending=False)

        # now I remove seeds from the output (they're included in the final lexicon regardless)
        threat_trial_df= threat_trial_df[~threat_trial_df['is_seed']]
        threat_trial_df= threat_trial_df.drop(columns=['is_seed', 'dimension'])

        # + drop sentence fragment bigrams as they're clearly artefacts & not meaningful bigrams
        threat_trial_df= threat_trial_df[~threat_trial_df['term'].str.contains(r'\b(and|to)\b', case=False, na=False)]
        threat_trial_df["securitising?"]= ""

        print(f"\nThe threat percentile we're working with right now is {thr_p}")
        print(f"Cutoff is {percentile_cutoff:.4f}")
        print(f"which added {len(candidates_above_cutoff)} candidates")
        print(f"bringing the total size up to {len(threat_trial_df)}")

        preview_df= pd.concat([threat_trial_df.head(15), threat_trial_df.tail(15)])
        print("here's a preview of the top & bottom 15:")
        display(preview_df[["term", "cosine_similarity", "securitising?"]])

        sheet_label= f"p_{str(thr_p).replace('.', '_')}"
        threat_trial_df.to_excel(excel_writer, sheet_name=sheet_label, index=False)

print(f"\ngreat, threat trials exported to {threat_trials_output_path}")

*Notes from my threshold exploration experimenting:*
At the 0.99 cosine similarity threshold, 36% of the bottom 50 candidate terms (i.e. the terms nearest the cutoff) have been manually validated as securitising threat language. Terms were coded as securitsing when they framed phenomena as either a severe or an exisitential threat linked to:
- organised violence
- terrorism
- warfare
- national security
and this framing can imply either protective or coercive state response in context.

Lower thresholds, particiarly 0.985, introduced substational drift into general crime and social harm discourse, which reduced concpetual precision. Whilst the 0.995 threshold did yeild higher precision, the terms from that tier on its own were drastically limited in scope.

In [ ]:
# then i load the manually vetted threat terms from the 99th percentile sheet
threat_expanded_terms= loadin_vetted_dimension_terms(dimension="threat", excel_file="threat_percentile_trials.xlsx", sheet_name="p_0_99")

manual_threat_term_additions= {"firearm", "firearms", "a weapon", "paramilitary groups", "zombie", "extremism", "suffering", "regimes", "regime"}
threat_expanded_terms= threat_expanded_terms.union(manual_threat_term_additions)

display(threat_expanded_terms)

My manual additions to the threat lexicon candidates are below.
These are mostly lexical repairs where a term is clearly securitising once corrected.

- "a weapon": from OCR artefact "aweapon" - contextual signal, e.g. "a weapon of mass destruction"
- "paramilitary groups": fixing OCR artefact "paramilitarygroups"
- "zombie": from "lethal zombieof"; comes up in drug-related securitising contexts
- "extremism": from "extremism increasing" and "from extremism"
- "genocide" and "gaza": from "genocide gaza"
- "countercyber": from "around countercyber" and "countercyber attacks"
- "sexporting": from "sexporting of" and "industry sexporting"
- "conflicts": from "of conflicts"
- "terrorism": from "terrorism of" and "brutal terrorism"
- "islamist": from "of islamist"
- "suffering": from "suffering accross"
- "regime/s": present in another percentile

# b. Urgency
In a similar manner, urgency in a securitisation context must now be defined in order to perform proper filtering. I already did this in the previous notebook where I define the seed lexcions, but I'm re-iterating it here for consistency's sake.

This definition (drawing very much so from the Copenhagen School official securitisation literature) distinguishes general time language, emotional intensity and routine deadlines from actual securitsing urgency. In other words, urgency isn't about speed but about the collapse of nromal political temporality.

The urgency percentile lists contain temporally intense or priority-related language that's sematnically similar to the Urgency seed lexicon, but filtering is to ensure they function in a securitisation context, i.e. they contribute to compresssing political time such that normal deliberative processes are framed as dangerously inadequate.

TLDR:
urgency is the mechanism through which an issue is moved from:

**routine politics --> emergency politics**.

Therefore, for a uni/bigram to be accepted into the expanded Urgency lexicon, it must be:


*   Temporally compressive (not only referring to time)
*   Politically consequential (not adminstrative)
*   & plausible in a speech act that frames delay as catastrophic or unnaceptable.

The definition taken here is:
*Lexical items that construct delay as interolerable by presenting the decision window as rapidly closing or already exhausted, thereby justifying acceleration beyond normal pacing.*


Examples of rejected terms:
*   "long waits" (administrative termporal term)
*   "whole industries" - can qualify in collapse framing, but insufficient on its own


Examples of accepted terms:
*   "act urgently"
*   "immense uncertainty"



In [ ]:
urgency_exploration_percentiles= lexicon_expansion_config["urgency_exploration_percentiles"]
urgency_trials_output_path= "urgency_percentile_trials.xlsx"

# all same as before but for urgency
with pd.ExcelWriter(urgency_trials_output_path, engine="xlsxwriter") as excel_writer:
    urgency_dim_df= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == "urgency"].copy()
    urgency_seed_terms= urgency_dim_df[urgency_dim_df["is_seed"]]
    urgency_candidate_terms= urgency_dim_df[~urgency_dim_df["is_seed"]]

    for urge_p in urgency_exploration_percentiles:
        percentile_cutoff= urgency_candidate_terms["cosine_similarity"].quantile(urge_p)
        candidates_above_cutoff= urgency_candidate_terms[urgency_candidate_terms["cosine_similarity"] >= percentile_cutoff].sort_values("cosine_similarity", ascending=False)

        urgency_trial_df= pd.concat([urgency_seed_terms, candidates_above_cutoff],ignore_index=True).sort_values("cosine_similarity", ascending=False)

        urgency_trial_df= urgency_trial_df[~urgency_trial_df['is_seed']]
        urgency_trial_df= urgency_trial_df.drop(columns=['is_seed', 'dimension'])
        urgency_trial_df= urgency_trial_df[~urgency_trial_df['term'].str.contains(r'\b(and|to)\b', case=False, na=False)]
        urgency_trial_df["securitising?"]= ""

        print(f"\n Current urgency percentile is {urge_p}")
        print(f"with a cutoff at {percentile_cutoff:.4f}")
        print(f"causing {len(candidates_above_cutoff)} candidates to be added for review")
        print(f"& bringing the total size to {len(urgency_trial_df)}")

        preview_df= pd.concat([urgency_trial_df.head(15), urgency_trial_df.tail(15)])
        display(preview_df[["term", "cosine_similarity", "securitising?"]])

        sheet_label= f"p_{str(urge_p).replace('.', '_')}"
        urgency_trial_df.to_excel(excel_writer, sheet_name=sheet_label, index=False)

print(f"\nurgency trials exported to {urgency_trials_output_path}")

Urgency terms flagged as semantically similar to the Urgency seed list were significantly less aligned with its securitisation definition that Threat was. Intuitivley, this makes sense; Threat is much more conceptually concrete and lexically clustered, whereas by contrast, urgency is highly dependent on surrounding ocntext and more often more subtle. Temporal compression is more often conveyed through metaphors, adverbs, modal constructions and syntactic framing than nouns, so terms like "before its too late" and "time is running out" are less semantically tight.

Overall, it seems embedding similarty struggles due to this, as the resulting words are polluted by procedural urgency, bureaucractic priority and routine policy acceleration. i.e. cosine similarity is better at finding semantic fields (common in threat) than performative discursive functions (common in urgency).

Therefore I chose a 0.995 thresho, which gives me a candiate pool that's sufficiently aligned with the theoretical definition of urgency as a collapse of political temporality.

In [ ]:
urgency_expanded_terms= loadin_vetted_dimension_terms(dimension="urgency", excel_file="urgency_percentile_trials.xlsx", sheet_name="p_0_995")

manual_urgency_term_additions= {"national crisis", "wakeup call", "degrading window", "repressive regimes", "oppressive regime", "oppressive regimes"}
urgency_expanded_terms= urgency_expanded_terms.union(manual_urgency_term_additions)

display(urgency_expanded_terms)

these are my manual additions to the urgency lexicon candidates:

- "national crisis": from OCR artefact "anational crises" - bigram still securitising when corrected
- "wakeup call": stems from "awake call"
- "degrading window": used in similar context to "degrading corridor"
- "urgent" and "urgently": generalise "act urgently", "just urgently", "changes urgent"
- "repressive regimes", "oppressive regime", "oppressive regimes": pair nicely with "repressive regime"

# c. Justification of Extraordinary Power

Now justification in a securitisation context is to be defined here for filtering.

A securitising move constructs a threat as urgent to legitimate the suspension, curcmvention or reshaping of normal political rules per the Copenhagen School.

I define it as:
*Terms that legitimise or presuppose the necessity of departing from ordinary political procedures, legal constraints or institutional norms to address a constructed threat.*

To be accepted, uni/bigrams must be:
*   Procedurally exceptional (not policy-oriented).
*   Legitimising of rule modification.
*   Plausaible in a speech act that authorises bypassing ordinary democratic constraints.

In other words, candidate terms must either suspend normal procedures, expand emergency authority, curtail oversight/deliberation or override existing constraints through percieved necessity.

Terms advocating reform, signalling strong leadership, proposing new policies or expressing determination do not count.

In [ ]:
# note: i'm using higher percentiles here due to the disproportionate weighting of terms
# most commonly aligning with "justification", to keep human review feasible
justification_exploration_percentiles= lexicon_expansion_config["justification_exploration_percentiles"]
justification_trials_output_path= "justification_percentile_trials_higher_percentiles.xlsx"

# & again, same as before
with pd.ExcelWriter(justification_trials_output_path, engine="xlsxwriter") as excel_writer:
    just_dim_df= best_dimension_per_term_df[best_dimension_per_term_df["dimension"] == "justification"].copy()
    just_seed_terms= just_dim_df[just_dim_df["is_seed"]]
    just_candidate_terms= just_dim_df[~just_dim_df["is_seed"]]

    for just_p in justification_exploration_percentiles:
        percentile_cutoff= just_candidate_terms["cosine_similarity"].quantile(just_p)

        candidates_above_cutoff= just_candidate_terms[just_candidate_terms["cosine_similarity"] >= percentile_cutoff].sort_values("cosine_similarity", ascending=False)
        just_trial_df= pd.concat([just_seed_terms, candidates_above_cutoff],ignore_index=True).sort_values("cosine_similarity", ascending=False)

        just_trial_df= just_trial_df[~just_trial_df['is_seed']]
        just_trial_df= just_trial_df.drop(columns=['is_seed', 'dimension'])
        just_trial_df= just_trial_df[~just_trial_df['term'].str.contains(r'\b(and|to)\b', case=False, na=False)]
        just_trial_df["securitising?"]= ""

        print(f"\nCurrent justification percentile is {just_p}")
        print(f"with a cutoff of {percentile_cutoff:.4f}")
        print(f"causing {len(candidates_above_cutoff)} candidates to be added")
        print(f"& bringing the justification trial df up to {len(just_trial_df)}")

        preview_df= pd.concat([just_trial_df.head(15), just_trial_df.tail(15)])
        display(preview_df[["term", "cosine_similarity", "securitising?"]])

        sheet_label= f"p_{str(just_p).replace('.', '_')}"
        just_trial_df.to_excel(excel_writer, sheet_name=sheet_label, index=False)

print(f"\njustification trials exported to {justification_trials_output_path}")

Unlike threat, I found no clear monotonic improvment in conceptual precision as the percentile increases. Even terms in the last 0.001 percentile still contain many topical security words and descriptive nouns, lacking in clear procedural rupte terms. This suggests that extraordinary power justification is not lexcially cohesive in embedding space.

Conceptually, this could be explained by justification of exception being clause-level, rhetorical, highly contextual and often realised through modal constructions.

For example, a term like "whatever it takes" or "without parlimentary delay" would not be picked up through this method as they're constructions, not simple collocations.

In [ ]:
justification_expanded_terms= loadin_vetted_dimension_terms(dimension="justification", excel_file="justification_percentile_trials.xlsx", sheet_name="p_0_997")

manual_justification_term_additions= {"staunch defense"}
justification_expanded_terms= justification_expanded_terms.union(manual_justification_term_additions)

display(justification_expanded_terms)

these are my manual additions to the justification lexicon candidates:

- "humanitarian intervention": taken from the 0.997 percentile list, particularly salient for justification
- "staunch defense": from OCR artefact "astaunch defense"
'''


# Grading
I now grade the terms into primary, secondary & contextual signals according to the same rules used to grade the seed lists (see my last notebook for that).

In [ ]:
# now I categorise the threat expanded terms into tiers
print(threat_expanded_terms)

threat_core_signals= {"extremism", "genocide", "gaza", "devastating power", "attack on", "invasion of", "to defeat", "bomb", "daesh", "bombing of", "linked terror", "brutal invasion",
                      "inexcusable violence", "punishment attacks", "terror movements", "terrorism", "lethal", "the assault", "hybrid warfare", "islamist extremism", "violent extremism",
                      "islamic", "islamist", "conflicts violence", "further attacks"}
threat_secondary_signals= {"emerging threats", "paramilitaries", "repressive regime", "repressive regimes", "opressive regime", "oppressive regimes", "escalation of", "countercyber", "warfare mps", "prisoners of", "international terrorism", "repressive",
                           "toxic legacies", "fuel conflicts", "islamist", "a weapon", "crime gangs", "conflicts", "atrocities", "security priorities", "aegis", "arm industry", "paramilitary groups",
                           "has prevailed", "our threat", "arms industry"}
threat_contextual_signals= {"sexporting", "prevailed", "soft generation", "suffering", "the innocent", "of helplessness", "day slavery", "being radicalised", "troubles were", "zombie"}

# & these are any terms not yet assigned to a tier, which are useful just for a sanity check if I wanted it
print(threat_expanded_terms.difference(threat_core_signals, threat_secondary_signals, threat_contextual_signals))

print("\n")
print(urgency_expanded_terms)
urgency_core_signals= {"urgent", "urgently", "urgency", "action now", "national crisis"}
urgency_secondary_signals= {"momentum", "wakeup call", "threatening crisis", "requires action", "end degrading"}
urgency_contextual_signals= {"irreversible", "within hours", "overrun", "degrading corridor", "degrading window"}
print(urgency_expanded_terms.difference(urgency_core_signals, urgency_secondary_signals, urgency_contextual_signals))

print("\n")
print(justification_expanded_terms)
justification_core_signals= {"freedom of", "further attack", "staunch defense", "war crimes", "police powers"}
justification_secondary_signals= {"security of", "protecting people", "investigation measures", "proportionate measures", "preserving the", "take proportionate", "protect frontline", "upholding the", "humanitarian intervention",
                                  "will resist", "to protect", "anti terrorism", "threats posed", "terrorism prevention", "to safeguard", "keep safe", "to peace"}
justification_contextual_signals= {"force border"}
print(justification_expanded_terms.difference(justification_core_signals, justification_secondary_signals, justification_contextual_signals))

In [ ]:
# I created this little helper to convert graded signal sets into a standardised df to make analysis cleaner
def build_signals_dataframe(core_signals, secondary_signals, contextual_signals):
    signal_rows= []
    for term in core_signals:
        signal_rows.append({"Term": term, "Signal Type": "core"})
    for term in secondary_signals:
        signal_rows.append({"Term": term, "Signal Type": "secondary"})
    for term in contextual_signals:
        signal_rows.append({"Term": term, "Signal Type": "contextual"})
    return pd.DataFrame(signal_rows)

# & this helper to merge my original seed lexicon with new expanded signals, deduplicating in favour of the new entries
def merge_and_deduplicate_lexicon(original_lexicon_df, new_signals_df):
    combined_df= pd.concat([original_lexicon_df, new_signals_df], ignore_index=True)

    # normalise before deduplication
    combined_df['Term']= combined_df['Term'].astype(str).str.lower().str.strip()
    combined_df['Signal Type']= combined_df['Signal Type'].astype(str).str.lower().str.strip()

    # but keep the last (i.e. newest) entry on duplicate terms
    deduped_df= combined_df.drop_duplicates(subset=['Term'], keep='last')
    return deduped_df


seed_lexicon_path= lexicon_expansion_config["seed_lexicon_path"]
updated_lexicon_output_path= "sbert_updated_lexicons.xlsx"

if not os.path.exists(seed_lexicon_path):
    print(f"couldn't find the lexcion at {seed_lexicon_path}, check that")
else:
    try:
        print(f"alright, reading original data from {seed_lexicon_path}...")

        original_lexicon_file= pd.ExcelFile(seed_lexicon_path)
        original_threat_df= pd.read_excel(original_lexicon_file, 'Threat Lexicon')
        original_urgency_df= pd.read_excel(original_lexicon_file, 'Urgency Lexicon')
        original_just_df= pd.read_excel(original_lexicon_file, 'Justification Lexicon')

        expanded_threat_signals_df= build_signals_dataframe(threat_core_signals, threat_secondary_signals, threat_contextual_signals)
        expanded_urgency_signals_df= build_signals_dataframe(urgency_core_signals, urgency_secondary_signals, urgency_contextual_signals)
        expanded_just_signals_df= build_signals_dataframe(justification_core_signals, justification_secondary_signals, justification_contextual_signals)

        print("merging & standardising lexicons...")
        final_threat_lexicon_df= merge_and_deduplicate_lexicon(original_threat_df, expanded_threat_signals_df)
        final_urgency_lexicon_df= merge_and_deduplicate_lexicon(original_urgency_df, expanded_urgency_signals_df)
        final_just_lexicon_df= merge_and_deduplicate_lexicon(original_just_df, expanded_just_signals_df)

        signal_tier_order= ['core', 'secondary', 'contextual']
        for lexicon_df in [final_threat_lexicon_df, final_urgency_lexicon_df, final_just_lexicon_df]:
            lexicon_df['Signal Type']= pd.Categorical(lexicon_df['Signal Type'], categories=signal_tier_order, ordered=True)
            lexicon_df.sort_values(by=['Signal Type', 'Term'], ascending=[True, True], inplace=True)

        print(f"writing to {updated_lexicon_output_path}...")
        with pd.ExcelWriter(updated_lexicon_output_path, engine='openpyxl') as excel_writer:
            final_threat_lexicon_df.to_excel(excel_writer, sheet_name='Threat Lexicon', index=False)
            final_urgency_lexicon_df.to_excel(excel_writer, sheet_name='Urgency Lexicon', index=False)
            final_just_lexicon_df.to_excel(excel_writer, sheet_name='Justification Lexicon', index=False)

        print(f"great, file saved as {updated_lexicon_output_path}")
        print(f"threat terms are these: {len(final_threat_lexicon_df)}")
        print(f"& urgency ones these: {len(final_urgency_lexicon_df)}")
        print(f"& finally justification ones: {len(final_just_lexicon_df)}")

    except Exception as lex_read_e:
        print(f"there was a problem there, use this for debugging: {lex_read_e}")

### My reflections + future work suggestions
The above results suggest that alone, semantic similarity on uni and bi grams alone aren't *fully* sufficient for robust expansion of the lexicons, particarly in the Urgency and Justification of Extraordinary Power domains. I needed to carefully manually vet everything, even when choosing a very conservative threshold for each dimension.

Therefore, future work could adopt a hybrid approach combining SBERT semantic filtering (to ensure domain relevance) with functional pattern detection to identify urgent and justificatory constructions.

There patterns could include:
*   Modal obligation structures (e.g. must, necessary to, required to)
*   Restricttive or prohivbitive constructions (e.g. ban, outlaw)
*   Authorisation language (e.g. grant powers, emergency measures)
*   Legitimising proportionality framing (e.g. proportionate measures, in order to prodct)
*   tri-grams

This approach could treat similarity as a first-stage filtering mechanism, then apply rule-based or supervised classification to expand the methodology to be sensitive to not only topic similarity but to securitisation-specific performative logic too.